In [2]:
import torch
import torch.nn as nn
from google.colab import drive

drive.mount('/content/drive')
!cp -r /content/drive/MyDrive/dataset_LSA-T/motion_features ./
!cp -r /content/drive/MyDrive/dataset_LSA-T/spatial_features ./
!cp /content/drive/MyDrive/dataset_LSA-T/meta_with_splits.csv ./

Mounted at /content/drive
cp: cannot stat '/content/drive/MyDrive/dataset_LSA-T/motion_features': No such file or directory
cp: cannot stat '/content/drive/MyDrive/dataset_LSA-T/spatial_features': No such file or directory


In [3]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import LoraConfig, TaskType, get_peft_model
def load_flan_llm(model_name="google/flan-t5-base", device="cuda"):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    hidden_size = base_model.config.d_model
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q", "v"],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.SEQ_2_SEQ_LM
    )

    flan_llm = get_peft_model(base_model, lora_config)
    flan_llm.to(device)
    flan_llm.print_trainable_parameters()

    return flan_llm, tokenizer,hidden_size

In [4]:
class SpaMoAdapter(nn.Module):
    def __init__(self, spatial_dim=1024, motion_dim=1024, llm_hidden_size=768):
        super().__init__()

        self.spatial_projection = nn.Linear(in_features=spatial_dim, out_features=llm_hidden_size)
        self.motion_projection  = nn.Linear(in_features=motion_dim, out_features=llm_hidden_size)

        self.conv1d_sequence = nn.Sequential(
            nn.Conv1d(llm_hidden_size, llm_hidden_size, kernel_size=5, padding=2),
            nn.MaxPool1d(kernel_size=2, stride=2, padding=0),
            nn.ReLU(),
            nn.Conv1d(llm_hidden_size, llm_hidden_size, kernel_size=5, padding=2),
            nn.MaxPool1d(kernel_size=2, stride=2, padding=0),
            nn.ReLU()
        )

        self.encoder_decoder_adapter = nn.Sequential(
            nn.Linear(llm_hidden_size, 4 * llm_hidden_size),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(4 * llm_hidden_size, llm_hidden_size),
            nn.Dropout(0.1)
        )

    def forward(self, spatial_feats, motion_feats):
        spatial_proj = self.spatial_projection(spatial_feats)
        motion_proj  = self.motion_projection(motion_feats)

        if spatial_proj.shape[1] != motion_proj.shape[1]:
            motion_proj = motion_proj.repeat(1, spatial_proj.shape[1], 1)

        combined = spatial_proj + motion_proj

        combined = combined.transpose(1, 2)
        combined = self.conv1d_sequence(combined)
        combined = combined.transpose(1, 2)

        output = self.encoder_decoder_adapter(combined)
        return output

In [5]:
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

class SignLanguageDataset(Dataset):
    def __init__(self, meta_df, spatial_dir, motion_dir):
        self.meta = meta_df.reset_index(drop=True)
        self.spatial_dir = spatial_dir
        self.motion_dir = motion_dir

    def __len__(self):
        return len(self.meta)

    def __getitem__(self, idx):
        row = self.meta.iloc[idx]
        video_id = row['id']
        text_target = row['label']

        spatial_path = os.path.join(self.spatial_dir, f"{video_id}.npy")
        motion_path = os.path.join(self.motion_dir, f"{video_id}.npy")

        spatial_feat = np.load(spatial_path)
        motion_feat = np.load(motion_path)

        return {
            "video_id": video_id,
            "spatial": torch.tensor(spatial_feat, dtype=torch.float32),
            "motion": torch.tensor(motion_feat, dtype=torch.float32),
            "target_text": text_target
        }

In [6]:
import math
from dataclasses import dataclass
import pandas as pd
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader


@dataclass
class TrainingConfig:
    epochs: int
    best_val_loss: float
    best_model_state: object
    optimizer: torch.optim.Optimizer
    criterion: nn.Module
    scheduler: LambdaLR


def get_spa_mo_scheduler(optimizer, warmup_steps, total_steps, lr_max=1e-4, lr_min=5e-5):
    def lr_lambda(current_step: int):
        if current_step < warmup_steps:
            return float(current_step) / float(max(1, warmup_steps))

        progress = float(current_step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        progress = min(1.0, max(0.0, progress))
        min_ratio = lr_min / lr_max
        cosine_decay = 0.5 * (1.0 + math.cos(math.pi * progress))

        return min_ratio + (1.0 - min_ratio) * cosine_decay

    return LambdaLR(optimizer, lr_lambda)


def load_dataloaders(csv_path="meta_with_splits.csv"):
    meta = pd.read_csv(csv_path)
    meta_train  = meta[meta['split'] == 'train'].copy()
    meta_val    = meta[meta['split'] == 'val'].copy()
    meta_test   = meta[meta['split'] == 'test'].copy()

    spatial_dir = './spatial_features_lsat'
    motion_dir = './motion_features_lsat'

    val_dataset     = SignLanguageDataset(meta_val, spatial_dir, motion_dir)
    train_dataset   = SignLanguageDataset(meta_train, spatial_dir, motion_dir)
    test_dataset    = SignLanguageDataset(meta_test, spatial_dir, motion_dir)

    train_loader = DataLoader(
        train_dataset,
        batch_size=64,
        shuffle=True,
        num_workers=2
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=64,
        shuffle=False,
        num_workers=2
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=64,
        shuffle=True,
        num_workers=2
    )

    return train_loader,val_dataset,test_loader


def load_initial_configuration(trainable_params, total_steps, epochs=40):
    best_val_loss = float('inf')
    best_model_state = None

    optimizer = torch.optim.AdamW(
        trainable_params,
        lr=1e-4,
        betas=(0.9, 0.98),
        weight_decay=0.01
    )

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    warmup_steps = min(10000, int(total_steps * 0.1))

    scheduler = get_spa_mo_scheduler(
        optimizer,
        warmup_steps=warmup_steps,
        total_steps=total_steps,
        lr_max=1e-4,
        lr_min=5e-5
    )

    return TrainingConfig(
        epochs=epochs,
        best_val_loss=best_val_loss,
        best_model_state=best_model_state,
        optimizer=optimizer,
        criterion=criterion,
        scheduler=scheduler
    )



def train_one_epoch(device, train_loader, flan_llm, tokenizer, adapter, config):
    adapter.train()
    flan_llm.train()
    running_loss = 0.0

    for batch in train_loader:
        config.optimizer.zero_grad()

        spatial_feats = batch['spatial'].to(device)
        motion_feats  = batch['motion'].to(device)
        target_texts  = batch['target_text']

        visual_embeds = adapter(spatial_feats, motion_feats)

        labels = tokenizer(
                target_texts,
                padding=True,
                truncation=True,
                return_tensors="pt"
            ).input_ids.to(device)

        labels[labels == tokenizer.pad_token_id] = -100

        outputs = flan_llm(
                inputs_embeds=visual_embeds,
                labels=labels
            )

        loss = outputs.loss
        loss.backward()

        config.optimizer.step()
        config.scheduler.step()

        running_loss += loss.item()
    return running_loss


def validate_one_epoch(device, val_loader, flan_llm, tokenizer, adapter):
    adapter.eval()
    flan_llm.eval()
    running_loss = 0.0

    with torch.no_grad():
        for batch in val_loader:
            spatial_feats = batch['spatial'].to(device)
            motion_feats  = batch['motion'].to(device)
            target_texts  = batch['target_text']

            visual_embeds = adapter(spatial_feats, motion_feats)

            labels = tokenizer(
                target_texts,
                padding=True,
                truncation=True,
                return_tensors="pt"
            ).input_ids.to(device)

            labels[labels == tokenizer.pad_token_id] = -100

            outputs = flan_llm(
                inputs_embeds=visual_embeds,
                labels=labels
            )

            running_loss += outputs.loss.item()

    return running_loss / len(val_loader)

def get_best_model(config, current_val_loss, adapter, flan_llm, save_path="spamo_best_checkpoint.pth"):
    if current_val_loss < config.best_val_loss:
        config.best_val_loss = current_val_loss
        config.best_model_state = {
            'adapter': adapter.state_dict(),
            'llm_lora': flan_llm.state_dict()
        }
        torch.save(config.best_model_state, save_path)
        print(f"   ↳ [NUEVO MEJOR] Guardado checkpoint con Val Loss: {current_val_loss:.4f}")

    return config.best_model_state, config.best_val_loss
def train_model(device,flan_llm, tokenizer,llm_hidden_size):
    epochs = 15
    train_loader,val_loader, test_loader = load_dataloaders()
    adapter = SpaMoAdapter(spatial_dim=1024, motion_dim=1024, llm_hidden_size=llm_hidden_size).to(device)
    trainable_params = list(adapter.parameters()) + list(flan_llm.parameters())
    total_steps = len(train_loader) * epochs
    config = load_initial_configuration(trainable_params, total_steps, epochs=epochs)
    train_losses,val_losses = [],[]
    for epoch in range(config.epochs):
        running_loss        = train_one_epoch(device, train_loader, flan_llm, tokenizer, adapter, config)
        current_val_loss    = validate_one_epoch(device,val_loader,flan_llm,tokenizer,adapter)
        current_train_loss    = running_loss / len(train_loader)
        best_model,best_val = get_best_model(config,current_val_loss,adapter,flan_llm)
        train_losses.append(current_train_loss)
        val_losses.append(current_val_loss)
        print(f"Epoch {epoch + 1}/{config.epochs} | Train Loss: {current_train_loss:.4f} | Val loss {current_val_loss:.4f}")
    return config.best_model_state,config.best_val_loss,train_losses,val_losses

In [ ]:
def load_cuda():
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Usando: {device}")
    return device

import matplotlib.pyplot as plt
import torch
import nltk
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

# Asegurar recursos de NLTK
nltk.download('punkt', quiet=True)


def plot_loss(train_losses, val_losses, output_path="loss_plot.png"):
    plt.figure(figsize=(8, 5))
    plt.plot(train_losses, label='Train Loss', color='#1f77b4', linewidth=2)
    plt.plot(val_losses, label='Validation Loss', color='#d62728', linewidth=2)
    plt.xlabel('Época')
    plt.ylabel('Pérdida (Loss)')
    plt.title('Curva de Pérdida de Entrenamiento y Validación')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.show()


def evaluate_bleu(model, test_loader, tokenizer, device):
    model.eval()
    references = []
    hypotheses = []

    smooth_fn = SmoothingFunction().method1

    with torch.no_grad():
        for batch in test_loader:
            inputs = batch['input'].to(device)
            targets = batch['target']  # Lista de oraciones o IDs de texto de referencia

            # Generar predicciones con el modelo (p. ej. greedy search o beam search)
            outputs = model.generate(inputs)

            # Decodificar tokens a texto si vienen como tensores
            if isinstance(outputs, torch.Tensor):
                pred_texts = tokenizer.batch_decode(outputs, skip_special_tokens=True)
            else:
                pred_texts = outputs

            if isinstance(targets, torch.Tensor):
                ref_texts = tokenizer.batch_decode(targets, skip_special_tokens=True)
            else:
                ref_texts = targets

            # Formatear para NLTK: referencias -> [[tokens]], hipótesis -> [tokens]
            for ref, pred in zip(ref_texts, pred_texts):
                references.append([ref.strip().split()])
                hypotheses.append(pred.strip().split())

    # Calcular BLEU-1 a BLEU-4
    b1 = corpus_bleu(references, hypotheses, weights=(1.0, 0, 0, 0), smoothing_function=smooth_fn)
    b2 = corpus_bleu(references, hypotheses, weights=(0.5, 0.5, 0, 0), smoothing_function=smooth_fn)
    b3 = corpus_bleu(references, hypotheses, weights=(0.33, 0.33, 0.33, 0), smoothing_function=smooth_fn)
    b4 = corpus_bleu(references, hypotheses, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth_fn)

    print("\n--- Resultados BLEU ---")
    print(f"BLEU-1: {b1 * 100:.2f}")
    print(f"BLEU-2: {b2 * 100:.2f}")
    print(f"BLEU-3: {b3 * 100:.2f}")
    print(f"BLEU-4: {b4 * 100:.2f}")

    return {"bleu1": b1, "bleu2": b2, "bleu3": b3, "bleu4": b4}


def main():
    device = load_cuda()
    adapter_output_path = "/content/drive/MyDrive/dataset_LSA-T/adapter_best_model.pth"
    llm_output_path = "/content/drive/MyDrive/dataset_LSA-T/llm_best_model.pth"
    _,_,test_loader = load_dataloaders()
    flan_llm, tokenizer,llm_hidden_size= load_flan_llm("google/flan-t5-base", device)
    best_model_state,best_val_loss,train_losses,val_losses = train_model(device,flan_llm, tokenizer,llm_hidden_size)
    torch.save(best_model_state.adapter.state_dict(), adapter_output_path)
    torch.save(best_model_state.llm.state_dict(), llm_output_path)
    plot_loss(train_losses, val_losses)

    evaluate_bleu(best_model_state, test_loader, tokenizer, device)


if __name__ == "__main__":
    main()